all required installs:

pip install imbalanced-learn
pip install geopandas
pip install pandas
pip install matplotlib
pip install scikit-learn

In [1]:
# import 

import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split


In [ ]:
# CSV Stuff

from pathlib import Path

DATA_PATH = Path('data/downSampBirdData.csv')
ZIP_PATH = Path('data/downSampBirdData.zip')

if not DATA_PATH.exists() and ZIP_PATH.exists():
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_PATH.parent)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH} or {ZIP_PATH}")

# Current project data file. Keep the old variable names so later cells still work.
df = pd.read_csv(DATA_PATH, dtype={'Streptopelia turtur': 'object'})
habitats_df = df.copy()


In [3]:
X = df.drop(columns=['Streptopelia turtur'])
y = df['Streptopelia turtur']

df['Streptopelia turtur'] = pd.to_numeric(df['Streptopelia turtur'], errors='coerce')

df['Streptopelia turtur'] = df['Streptopelia turtur'].fillna(0)

df['Streptopelia turtur'] = df['Streptopelia turtur'].apply(lambda x: 1 if x > 0 else 0)

X = df.drop(columns=['Streptopelia turtur'])
y = df['Streptopelia turtur']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

print("Distribution after downsampling (Trainingset):")
print(pd.Series(y_train_resampled).value_counts())

Distribution after downsampling (Trainingset):
Streptopelia turtur
0    120550
1     12055
Name: count, dtype: int64


In [4]:
df.to_csv('data/downSampBirdData.csv', index=False)

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

# Configure clean aesthetic presets for presentation exports
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'figure.titlesize': 16
})

# =========================================================
# 1. DATA STREAM ACQUISITION & INTEGRATION (Slide 2 & 5)
# =========================================================
print("--- Phase 1: Materializing Data Streams ---")

DATA_PATH = Path('data/downSampBirdData.csv')
ZIP_PATH = Path('data/downSampBirdData.zip')

if not DATA_PATH.exists() and ZIP_PATH.exists():
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_PATH.parent)

if 'df' in globals():
    observations_df = df.copy()
else:
    observations_df = pd.read_csv(DATA_PATH, dtype={'Streptopelia turtur': 'object'})

print(f"Tracking dataset loaded: {len(observations_df):,} records.")

# Cap this presentation/modeling cell so repeated notebook runs remain practical.
MAX_MODEL_ROWS = 250_000
if len(observations_df) > MAX_MODEL_ROWS:
    observations_df = observations_df.sample(n=MAX_MODEL_ROWS, random_state=42).copy()
    print(f"Using reproducible sample for modeling visuals: {MAX_MODEL_ROWS:,} rows.")

# =========================================================
# 2. FEATURE ENGINEERING FRAMEWORK (Slide 7)
# =========================================================
print("\n--- Phase 2: Applying Feature Engineering Framework ---")

observations_df['Streptopelia turtur'] = pd.to_numeric(
    observations_df['Streptopelia turtur'], errors='coerce'
).fillna(0)
observations_df['presence'] = (observations_df['Streptopelia turtur'] > 0).astype(int)

if 'eventDate' in observations_df.columns:
    event_dates = pd.to_datetime(observations_df['eventDate'], errors='coerce')
    observations_df['year'] = event_dates.dt.year
    observations_df['month'] = event_dates.dt.month
    observations_df['day_of_year'] = event_dates.dt.dayofyear

candidate_features = [
    'decimalLatitude',
    'decimalLongitude',
    'total_observations',
    'speciesgroup_observations',
    'year',
    'month',
    'day_of_year',
]
feature_cols = [col for col in candidate_features if col in observations_df.columns]

for col in feature_cols:
    observations_df[col] = pd.to_numeric(observations_df[col], errors='coerce')

merged_df = observations_df.dropna(subset=feature_cols + ['presence']).copy()
print(f"Master DataFrame 'merged_df' created successfully. Shape: {merged_df.shape[0]:,} rows.")
print(f"Features used: {', '.join(feature_cols)}")

if merged_df['presence'].nunique() < 2:
    raise ValueError('The modeled data contains only one target class; increase MAX_MODEL_ROWS or check the data.')

# =========================================================
# 3. EXPORT UNDERSTANDING GRAPHICS (Slide 3 & 4)
# =========================================================
print("\n--- Phase 3: Exporting Data Understanding Graphics ---")

plt.figure(figsize=(6, 4.5))
ax = sns.countplot(x='presence', data=merged_df, hue='presence', palette=['#bf616a', '#a3be8c'], legend=False)
plt.title("Target Distribution: The Accuracy Trap", pad=15)
plt.xlabel("Class Index")
plt.ylabel("Observations")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Absence", "Presence"])
plt.tight_layout()
plt.savefig('visual_slide3_imbalance.png', dpi=150)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
sns.histplot(merged_df['total_observations'], bins=30, ax=axes[0], color='#5e81ac', kde=True)
axes[0].set_title("Total Observation Distribution")
axes[0].set_xlabel("Total Observations")
sns.histplot(merged_df['speciesgroup_observations'], bins=30, ax=axes[1], color='#d08770', kde=True)
axes[1].set_title("Species Group Observation Distribution")
axes[1].set_xlabel("Species Group Observations")
plt.suptitle("Observation Effort Feature Distributions")
plt.tight_layout()
plt.savefig('visual_slide4_skewness.png', dpi=150)
plt.close()

# =========================================================
# 4. PREPARATION PIPELINE & SPLITTING (Slide 6 & Grader Feedback)
# =========================================================
print("\n--- Phase 4: Splitting Training Space to Eliminate Data Leakage ---")
X = merged_df[feature_cols]
y = merged_df['presence']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Pristine Testing Frame Size (Slide 6 & 8): {X_test.shape[0]:,} rows.")

rus_baseline = RandomUnderSampler(sampling_strategy='majority', random_state=42)
X_train_base, y_train_base = rus_baseline.fit_resample(X_train, y_train)

rus_optimized = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_train_opt, y_train_opt = rus_optimized.fit_resample(X_train, y_train)

# =========================================================
# 5. EXECUTE CLASSIFICATION ARCHITECTURES (Slide 8)
# =========================================================
print("\n--- Phase 5: Simulating Model Engines & Evaluating Minority Metrics ---")

rf_baseline = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
rf_baseline.fit(X_train_base, y_train_base)
y_pred_base = rf_baseline.predict(X_test)

rf_optimized = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
rf_optimized.fit(X_train_opt, y_train_opt)
y_pred_opt = rf_optimized.predict(X_test)

print("\n=== BASELINE MODEL METRICS (50/50 Strategy) ===")
print(classification_report(y_test, y_pred_base, target_names=['Absence', 'Presence'], zero_division=0))

print("=== OPTIMIZED FEEDBACK MODEL METRICS (1:10 Strategy) ===")
print(classification_report(y_test, y_pred_opt, target_names=['Absence', 'Presence'], zero_division=0))

def minority_precision(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    true_positive = matrix[1, 1]
    false_positive = matrix[0, 1]
    denominator = true_positive + false_positive
    return true_positive / denominator if denominator else 0.0

metrics_data = {
    'Strategy': ['Baseline (50/50)', 'Baseline (50/50)', 'Feedback Opt (1:10)', 'Feedback Opt (1:10)'],
    'Metric Type': ['Minority Precision', 'Minority F1-Score', 'Minority Precision', 'Minority F1-Score'],
    'Score': [
        minority_precision(y_test, y_pred_base),
        f1_score(y_test, y_pred_base, pos_label=1, zero_division=0),
        minority_precision(y_test, y_pred_opt),
        f1_score(y_test, y_pred_opt, pos_label=1, zero_division=0),
    ]
}
metrics_df = pd.DataFrame(metrics_data)

plt.figure(figsize=(7, 4.5))
sns.barplot(x='Strategy', y='Score', hue='Metric Type', data=metrics_df, palette='bone')
plt.title("Imbalance Mitigation Trade-offs: Baseline vs. Grader Feedback")
plt.ylabel("Performance Score")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.savefig('visual_slide8_comparison.png', dpi=150)
plt.close()

# =========================================================
# 6. EXTRACT FEATURE RULES (Slide 9)
# =========================================================
importances = rf_optimized.feature_importances_
feat_df = pd.DataFrame({'Predictive Feature': feature_cols, 'Weight': importances}).sort_values(by='Weight', ascending=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(x='Weight', y='Predictive Feature', data=feat_df, hue='Predictive Feature', palette='viridis', legend=False)
plt.title("Random Forest Structural Rules: Feature Importance Rankings")
plt.xlabel("Gini Importance Score")
plt.tight_layout()
plt.savefig('visual_slide9_importance.png', dpi=150)
plt.close()
print("\n--- Visual elements generated completely. Clean files exported to workspace directory. ---")


--- Phase 1: Materializing Data Streams ---


Tracking dataset loaded: 12,558,786 records.


Using reproducible sample for modeling visuals: 250,000 rows.

--- Phase 2: Applying Feature Engineering Framework ---
Master DataFrame 'merged_df' created successfully. Shape: 249,999 rows.
Features used: decimalLatitude, decimalLongitude, total_observations, speciesgroup_observations, year, month, day_of_year

--- Phase 3: Exporting Data Understanding Graphics ---



--- Phase 4: Splitting Training Space to Eliminate Data Leakage ---
Pristine Testing Frame Size (Slide 6 & 8): 50,000 rows.



--- Phase 5: Simulating Model Engines & Evaluating Minority Metrics ---



=== BASELINE MODEL METRICS (50/50 Strategy) ===
              precision    recall  f1-score   support

     Absence       1.00      0.87      0.93     49943
    Presence       0.01      0.89      0.02        57

    accuracy                           0.87     50000
   macro avg       0.50      0.88      0.47     50000
weighted avg       1.00      0.87      0.93     50000

=== OPTIMIZED FEEDBACK MODEL METRICS (1:10 Strategy) ===
              precision    recall  f1-score   support

     Absence       1.00      0.97      0.98     49943
    Presence       0.02      0.60      0.04        57

    accuracy                           0.97     50000
   macro avg       0.51      0.78      0.51     50000
weighted avg       1.00      0.97      0.98     50000




--- Visual elements generated completely. Clean files exported to workspace directory. ---
